# Super Voz F5-TTS - Kaggle

Execute em um kernel limpo com GPU e internet ligadas. O fluxo baixa `warllem/Super_voz`, valida o pacote F5-TTS e gera `/kaggle/working/resultado_voz.wav`. O log completo fica em `/kaggle/working/super_voz_kaggle.log`.


In [ ]:
# 1) Preparar ambiente, token e dependencias
from pathlib import Path
import os
import subprocess
import sys
import traceback

LOG_PATH = Path('/kaggle/working/super_voz_kaggle.log')
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
LOG_PATH.write_text('Super Voz F5-TTS Kaggle log\n', encoding='utf-8')

def log_error(stage, exc):
    with LOG_PATH.open('a', encoding='utf-8') as log_file:
        log_file.write(f'\n===== {stage} =====\n')
        log_file.write(''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)))
    print(f'Erro em {stage}. Log salvo em: {LOG_PATH}')

def run(command, check=True):
    print('Executando:', ' '.join(command))
    return subprocess.run(command, check=check, text=True)

def pip_install(*packages, extra_args=()):
    command = [sys.executable, '-m', 'pip', 'install', '-q', '--no-warn-conflicts', *extra_args, *packages]
    return run(command)

try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True, capture_output=True)
    if result.returncode == 0 and result.stdout.strip():
        print('GPU:', result.stdout.strip())
    else:
        print('Aviso: GPU nao detectada por nvidia-smi. A inferencia pode cair para CPU.')

    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for name in ('HF_TOKEN', 'HUGGINGFACE_TOKEN', 'HUGGING_FACE_HUB_TOKEN'):
            try:
                value = secrets.get_secret(name)
                if value:
                    os.environ[name] = value
                    print('Secret Hugging Face encontrado:', name)
                    break
            except Exception:
                pass
    except Exception as exc:
        print('Aviso: Kaggle Secrets indisponivel:', exc)

    if subprocess.run(['which', 'ffmpeg'], capture_output=True).returncode != 0:
        run(['apt-get', '-qq', 'update'])
        run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'])

    import numpy as np, scipy as sp, pandas as pd
    constraints_path = Path('/tmp/super_voz_constraints.txt')
    constraints_path.write_text(
        f'numpy=={np.__version__}\nscipy=={sp.__version__}\npandas=={pd.__version__}\n',
        encoding='utf-8',
    )

    packages = [
        'huggingface_hub>=0.31.0,<1.0',
        'hf-xet>=1.1.0',
        'f5-tts>=1.1.9',
        'safetensors>=0.4.5',
        'torch>=2.4.0',
        'torchaudio>=2.4.0',
        'soundfile>=0.12.1',
        'librosa>=0.10.1,<0.11.0',
        'pydub>=0.25.1',
        'vocos>=0.1.0',
        'hydra-core>=1.3.2',
        'omegaconf>=2.3.0',
        'cached-path>=1.5.1,<2.0.0',
        'transformers>=4.36.0,<5.0.0',
        'accelerate>=0.25.0',
        'gradio>=4.44.0',
    ]
    pip_install(*packages, extra_args=('-c', str(constraints_path)))

    check_code = """
import huggingface_hub, torch, torchaudio, soundfile, f5_tts
print('huggingface_hub', huggingface_hub.__version__)
print('torch', torch.__version__)
print('torchaudio', torchaudio.__version__)
print('soundfile ok')
print('f5_tts ok')
"""
    check = subprocess.run([sys.executable, '-c', check_code], text=True, capture_output=True)
    if check.returncode != 0:
        LOG_PATH.write_text('Falha na validacao de dependencias\n' + check.stdout + check.stderr, encoding='utf-8')
        raise RuntimeError(f'Falha ao validar dependencias. Log salvo em {LOG_PATH}')
    print(check.stdout)
except Exception as exc:
    log_error('celula 1 - ambiente e dependencias', exc)
    raise


In [ ]:
# 2) Criar modulo F5-TTS
from pathlib import Path
import sys
import traceback

LOG_PATH = Path('/kaggle/working/super_voz_kaggle.log')

def log_error(stage, exc):
    with LOG_PATH.open('a', encoding='utf-8') as log_file:
        log_file.write(f'\n===== {stage} =====\n')
        log_file.write(''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)))
    print(f'Erro em {stage}. Log salvo em: {LOG_PATH}')

try:
    MODULE_CODE = "from __future__ import annotations\n\nimport json\nimport logging\nimport math\nimport os\nimport re\nimport traceback\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\nfrom typing import Any\n\n\nHF_REPO_ID = \"warllem/Super_voz\"\nHF_REVISION = \"main\"\nPREFERRED_VOICE_PACKAGE = \"v_minha_voz_f5_tts_ptbr\"\nMODEL_ROOT = Path(os.environ.get(\"SUPER_VOZ_MODEL_ROOT\", \"/kaggle/working/Super_voz\"))\nOUTPUT_DIR = Path(os.environ.get(\"SUPER_VOZ_OUTPUT_DIR\", \"/kaggle/working\"))\nLOG_PATH = Path(os.environ.get(\"SUPER_VOZ_LOG_PATH\", \"/kaggle/working/super_voz_kaggle.log\"))\nTEST_TEXT = \"Boa noite Warllem, sua voz esta pronta.\"\n\nDEFAULT_F5TTS_V1_BASE_CONFIG: dict[str, Any] = {\n    \"backbone\": \"DiT\",\n    \"arch\": {\n        \"dim\": 1024,\n        \"depth\": 22,\n        \"heads\": 16,\n        \"ff_mult\": 2,\n        \"text_dim\": 512,\n        \"text_mask_padding\": True,\n        \"qk_norm\": None,\n        \"conv_layers\": 4,\n        \"pe_attn_head\": None,\n        \"attn_backend\": \"torch\",\n        \"attn_mask_enabled\": False,\n        \"checkpoint_activations\": False,\n    },\n    \"mel_spec\": {\n        \"target_sample_rate\": 24000,\n        \"n_mel_channels\": 100,\n        \"hop_length\": 256,\n        \"win_length\": 1024,\n        \"n_fft\": 1024,\n        \"mel_spec_type\": \"vocos\",\n    },\n    \"tokenizer\": \"char\",\n    \"vocoder\": \"vocos\",\n}\n\nPORTUGUESE_REQUIRED_CHARS = set(\"abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ\u00e1\u00e0\u00e2\u00e3\u00e9\u00ea\u00ed\u00f3\u00f4\u00f5\u00fa\u00fc\u00e7\u00c1\u00c0\u00c2\u00c3\u00c9\u00ca\u00cd\u00d3\u00d4\u00d5\u00da\u00dc\u00c7 .,!?;:-'\\\"\")\n\n\n@dataclass(frozen=True)\nclass RemoteFile:\n    path: str\n    size: int | None = None\n    lfs_size: int | None = None\n    oid: str | None = None\n    lfs_oid: str | None = None\n\n    @property\n    def expected_size(self) -> int | None:\n        return self.lfs_size or self.size\n\n\n@dataclass\nclass CheckpointChoice:\n    remote_path: str\n    reason: str\n    expected_size: int | None = None\n\n\n@dataclass\nclass VoicePackage:\n    repo_id: str\n    revision: str\n    root: Path\n    voice_dir: str\n    manifest_path: Path\n    manifest: dict[str, Any]\n    checkpoint: CheckpointChoice\n    vocab_remote_path: str\n    base_vocab_remote_path: str | None\n    reference_audio_remote_path: str\n    reference_text_remote_path: str | None\n    base_library_dir: str | None\n    setting_remote_path: str | None\n    config: dict[str, Any]\n    remote_files: dict[str, RemoteFile]\n    local_checkpoint_path: Path | None = None\n    local_vocab_path: Path | None = None\n    local_base_vocab_path: Path | None = None\n    local_reference_audio_path: Path | None = None\n    local_reference_text_path: Path | None = None\n    local_setting_path: Path | None = None\n\n\n@dataclass\nclass DiagnosticReport:\n    rows: list[tuple[str, str, str]] = field(default_factory=list)\n\n    def add(self, item: str, ok: bool, detail: str = \"\") -> None:\n        self.rows.append((item, \"OK\" if ok else \"FALHA\", detail))\n\n    @property\n    def ready(self) -> bool:\n        blocking = {\n            \"Manifesto\",\n            \"Checkpoint principal\",\n            \"Checkpoint legivel\",\n            \"Arquitetura identificada\",\n            \"Vocab encontrado\",\n            \"Vocab compativel\",\n            \"Referencia de audio\",\n            \"Vocoder\",\n        }\n        return all(status == \"OK\" for item, status, _ in self.rows if item in blocking)\n\n    def render(self) -> str:\n        lines = [f\"{'Item':<32} {'Status':<8} Detalhe\"]\n        for item, status, detail in self.rows:\n            lines.append(f\"{item:<32} {status:<8} {detail}\")\n        lines.append(f\"{'Inferencia pronta':<32} {'SIM' if self.ready else 'NAO':<8}\")\n        return \"\\n\".join(lines)\n\n\ndef setup_logging(log_path: Path = LOG_PATH) -> logging.Logger:\n    log_path.parent.mkdir(parents=True, exist_ok=True)\n    logger = logging.getLogger(\"super_voz_kaggle\")\n    logger.setLevel(logging.INFO)\n    logger.handlers.clear()\n\n    formatter = logging.Formatter(\"%(asctime)s %(levelname)s %(message)s\")\n    file_handler = logging.FileHandler(log_path, encoding=\"utf-8\")\n    file_handler.setFormatter(formatter)\n    stream_handler = logging.StreamHandler()\n    stream_handler.setFormatter(logging.Formatter(\"%(message)s\"))\n\n    logger.addHandler(file_handler)\n    logger.addHandler(stream_handler)\n    return logger\n\n\nLOGGER = setup_logging()\n\n\ndef log_exception_context(exc: BaseException, repo_id: str, revision: str, token: str | None) -> None:\n    LOGGER.error(\"Falha Hugging Face sem fallback silencioso.\")\n    LOGGER.error(\"Tipo da excecao: %s\", type(exc).__name__)\n    LOGGER.error(\"Mensagem: %s\", exc)\n    LOGGER.error(\"Repo ID: %s\", repo_id)\n    LOGGER.error(\"Revisao: %s\", revision)\n    LOGGER.error(\"Token presente: %s\", \"sim\" if token else \"nao\")\n    LOGGER.error(\"Traceback completo:\\n%s\", \"\".join(traceback.format_exception(type(exc), exc, exc.__traceback__)))\n\n\ndef get_kaggle_secret(name: str) -> str | None:\n    try:\n        from kaggle_secrets import UserSecretsClient\n\n        value = UserSecretsClient().get_secret(name)\n        return value or None\n    except Exception:\n        return None\n\n\ndef get_hf_token() -> str | None:\n    for name in (\"HF_TOKEN\", \"HUGGINGFACE_TOKEN\", \"HUGGING_FACE_HUB_TOKEN\"):\n        value = os.environ.get(name) or get_kaggle_secret(name)\n        if value:\n            return value\n    return None\n\n\ndef file_size_text(size: int | None) -> str:\n    if size is None:\n        return \"desconhecido\"\n    units = (\"B\", \"KB\", \"MB\", \"GB\", \"TB\")\n    value = float(size)\n    unit = units[0]\n    for unit in units:\n        if value < 1024 or unit == units[-1]:\n            break\n        value /= 1024\n    return f\"{value:.2f} {unit}\"\n\n\ndef list_remote_files(\n    repo_id: str = HF_REPO_ID,\n    revision: str = HF_REVISION,\n    token: str | None = None,\n) -> dict[str, RemoteFile]:\n    from huggingface_hub import HfApi\n\n    token = token or get_hf_token()\n    api = HfApi(token=token)\n    try:\n        tree = list(api.list_repo_tree(repo_id=repo_id, repo_type=\"model\", revision=revision, recursive=True, expand=True))\n    except Exception as exc:\n        log_exception_context(exc, repo_id, revision, token)\n        raise\n\n    remote_files: dict[str, RemoteFile] = {}\n    for item in tree:\n        path = getattr(item, \"path\", None)\n        if not path or item.__class__.__name__ == \"RepoFolder\":\n            continue\n        lfs = getattr(item, \"lfs\", None) or {}\n        lfs_size = lfs.get(\"size\") if isinstance(lfs, dict) else getattr(lfs, \"size\", None)\n        lfs_oid = lfs.get(\"oid\") if isinstance(lfs, dict) else (getattr(lfs, \"oid\", None) or getattr(lfs, \"sha256\", None))\n        remote_files[path] = RemoteFile(\n            path=path,\n            size=getattr(item, \"size\", None),\n            lfs_size=lfs_size,\n            oid=getattr(item, \"oid\", None) or getattr(item, \"blob_id\", None),\n            lfs_oid=lfs_oid,\n        )\n\n    LOGGER.info(\"Arquivos encontrados no Hugging Face (%s @ %s):\", repo_id, revision)\n    for file in sorted(remote_files.values(), key=lambda item: item.path):\n        LOGGER.info(\"- %s (%s)\", file.path, file_size_text(file.expected_size))\n    return remote_files\n\n\ndef download_remote_file(\n    remote_path: str,\n    output_dir: Path,\n    repo_id: str = HF_REPO_ID,\n    revision: str = HF_REVISION,\n    token: str | None = None,\n    expected_size: int | None = None,\n) -> Path:\n    from huggingface_hub import hf_hub_download\n\n    token = token or get_hf_token()\n    output_dir.mkdir(parents=True, exist_ok=True)\n    LOGGER.info(\"Baixando %s\", remote_path)\n    try:\n        local_path = Path(\n            hf_hub_download(\n                repo_id=repo_id,\n                repo_type=\"model\",\n                revision=revision,\n                filename=remote_path,\n                local_dir=str(output_dir),\n                token=token,\n            )\n        )\n    except TypeError:\n        local_path = Path(\n            hf_hub_download(\n                repo_id=repo_id,\n                repo_type=\"model\",\n                revision=revision,\n                filename=remote_path,\n                local_dir=str(output_dir),\n                token=token,\n            )\n        )\n    except Exception as exc:\n        log_exception_context(exc, repo_id, revision, token)\n        raise\n\n    validate_downloaded_file(local_path, expected_size)\n    return local_path\n\n\ndef validate_downloaded_file(path: Path, expected_size: int | None = None) -> None:\n    if not path.is_file():\n        raise FileNotFoundError(f\"Arquivo baixado nao encontrado: {path}\")\n    size = path.stat().st_size\n    if size <= 0:\n        raise RuntimeError(f\"Arquivo vazio: {path}\")\n    if expected_size is not None and size != expected_size:\n        raise RuntimeError(f\"Tamanho invalido em {path}: baixado={size}, esperado={expected_size}\")\n    LOGGER.info(\"Arquivo local: %s (%s)\", path, file_size_text(size))\n\n\ndef load_json(path: Path) -> dict[str, Any]:\n    return json.loads(path.read_text(encoding=\"utf-8\"))\n\n\ndef resolve_manifest_remote_path(manifest: dict[str, Any], manifest_path: str, value: str | None) -> str | None:\n    if not value:\n        return None\n    if value.startswith(\"voices/\") or value.startswith(\"libraries/\"):\n        return value\n    remote_dir = manifest.get(\"huggingface_remote_dir\") or str(Path(manifest_path).parent).replace(\"\\\\\", \"/\")\n    return f\"{remote_dir.rstrip('/')}/{value.lstrip('/')}\"\n\n\ndef choose_manifest(remote_files: dict[str, RemoteFile], root: Path, repo_id: str, revision: str, token: str | None) -> tuple[str, Path, dict[str, Any]]:\n    manifest_paths = sorted(path for path in remote_files if re.fullmatch(r\"voices/[^/]+/manifest\\.json\", path))\n    if not manifest_paths:\n        raise RuntimeError(\"Nenhum manifest.json F5-TTS encontrado em voices/*/manifest.json.\")\n\n    candidates: list[tuple[int, str, Path, dict[str, Any]]] = []\n    for remote_path in manifest_paths:\n        local_path = download_remote_file(remote_path, root, repo_id, revision, token, remote_files[remote_path].expected_size)\n        manifest = load_json(local_path)\n        score = 0\n        if manifest.get(\"architecture\") == \"F5-TTS\":\n            score += 100\n        if manifest.get(\"package_name\") == PREFERRED_VOICE_PACKAGE:\n            score += 50\n        if manifest.get(\"huggingface_remote_dir\", \"\").endswith(PREFERRED_VOICE_PACKAGE):\n            score += 30\n        if remote_path.startswith(f\"voices/{PREFERRED_VOICE_PACKAGE}/\"):\n            score += 20\n        if resolve_manifest_remote_path(manifest, remote_path, manifest.get(\"voice_checkpoint\")) in remote_files:\n            score += 10\n        candidates.append((score, remote_path, local_path, manifest))\n\n    score, remote_path, local_path, manifest = sorted(candidates, reverse=True)[0]\n    if manifest.get(\"architecture\") != \"F5-TTS\":\n        raise RuntimeError(f\"Manifesto escolhido nao e F5-TTS: {remote_path}\")\n    LOGGER.info(\"Manifesto escolhido: %s (score=%s)\", remote_path, score)\n    return remote_path, local_path, manifest\n\n\ndef choose_checkpoint(manifest: dict[str, Any], manifest_remote_path: str, remote_files: dict[str, RemoteFile]) -> CheckpointChoice:\n    policy = (\n        (\"voice_checkpoint\", \"checkpoint recomendado pelo manifest.json\"),\n        (\"inference_checkpoint\", \"checkpoint final de inferencia do manifest.json\"),\n        (\"final_checkpoint\", \"checkpoint final do manifest.json\"),\n        (\"latest_checkpoint\", \"checkpoint mais recente do manifest.json\"),\n    )\n    for key, reason in policy:\n        remote_path = resolve_manifest_remote_path(manifest, manifest_remote_path, manifest.get(key))\n        if remote_path and remote_path in remote_files and remote_path.endswith((\".pt\", \".safetensors\")):\n            return CheckpointChoice(remote_path, reason, remote_files[remote_path].expected_size)\n\n    voice_dir = manifest.get(\"huggingface_remote_dir\") or str(Path(manifest_remote_path).parent).replace(\"\\\\\", \"/\")\n    fallback_names = (\"model/model_last.safetensors\", \"model/model_last.pt\", \"model/latest_checkpoint.pt\")\n    for name in fallback_names:\n        remote_path = f\"{voice_dir.rstrip('/')}/{name}\"\n        if remote_path in remote_files:\n            return CheckpointChoice(remote_path, f\"fallback validado existente: {name}\", remote_files[remote_path].expected_size)\n\n    raise RuntimeError(\"Nenhum checkpoint F5-TTS valido encontrado pelo manifesto ou fallbacks F5.\")\n\n\ndef infer_reference_text_path(reference_audio_remote_path: str, remote_files: dict[str, RemoteFile]) -> str | None:\n    audio = Path(reference_audio_remote_path)\n    candidates = [\n        str(audio.with_suffix(\".txt\")).replace(\"\\\\\", \"/\"),\n        str(audio.parent / \"referencia_voz.txt\").replace(\"\\\\\", \"/\"),\n        str(audio.parent / \"reference.txt\").replace(\"\\\\\", \"/\"),\n    ]\n    for candidate in candidates:\n        if candidate in remote_files:\n            return candidate\n    return None\n\n\ndef get_f5_config(manifest: dict[str, Any], setting: dict[str, Any] | None = None) -> dict[str, Any]:\n    exp_name = manifest.get(\"exp_name\") or (setting or {}).get(\"exp_name\")\n    if exp_name != \"F5TTS_v1_Base\":\n        raise RuntimeError(\n            \"O repositorio nao contem configuracao completa para uma arquitetura diferente de F5TTS_v1_Base. \"\n            f\"exp_name encontrado: {exp_name!r}\"\n        )\n    config = json.loads(json.dumps(DEFAULT_F5TTS_V1_BASE_CONFIG))\n    config[\"tokenizer\"] = manifest.get(\"tokenizer\") or (setting or {}).get(\"tokenizer_type\") or \"char\"\n    if config[\"tokenizer\"] != \"char\":\n        raise RuntimeError(f\"Tokenizer nao suportado para esta voz: {config['tokenizer']}\")\n    return config\n\n\ndef build_voice_package(\n    repo_id: str = HF_REPO_ID,\n    revision: str = HF_REVISION,\n    root: Path = MODEL_ROOT,\n    token: str | None = None,\n    download_checkpoint: bool = True,\n) -> VoicePackage:\n    token = token or get_hf_token()\n    remote_files = list_remote_files(repo_id, revision, token)\n    manifest_remote_path, local_manifest_path, manifest = choose_manifest(remote_files, root, repo_id, revision, token)\n    voice_dir = manifest.get(\"huggingface_remote_dir\") or str(Path(manifest_remote_path).parent).replace(\"\\\\\", \"/\")\n\n    checkpoint = choose_checkpoint(manifest, manifest_remote_path, remote_files)\n    vocab_remote_path = f\"{voice_dir.rstrip('/')}/model/vocab.txt\"\n    if vocab_remote_path not in remote_files:\n        raise RuntimeError(f\"Vocabulario da voz nao encontrado: {vocab_remote_path}\")\n\n    base_library = manifest.get(\"base_library\") or {}\n    base_library_dir = base_library.get(\"huggingface_remote_dir\")\n    base_vocab_remote_path = f\"{base_library_dir}/vocab.txt\" if base_library_dir else None\n    if base_vocab_remote_path and base_vocab_remote_path not in remote_files:\n        LOGGER.warning(\"Vocabulario da biblioteca-base declarado, mas ausente: %s\", base_vocab_remote_path)\n        base_vocab_remote_path = None\n\n    reference_audio_remote_path = f\"{voice_dir.rstrip('/')}/data_reference/referencia_voz.wav\"\n    if reference_audio_remote_path not in remote_files:\n        raise RuntimeError(f\"Audio de referencia nao encontrado: {reference_audio_remote_path}\")\n    reference_text_remote_path = infer_reference_text_path(reference_audio_remote_path, remote_files)\n\n    setting_remote_path = f\"{base_library_dir}/setting.json\" if base_library_dir else None\n    local_setting_path = None\n    setting = None\n    if setting_remote_path and setting_remote_path in remote_files:\n        local_setting_path = download_remote_file(setting_remote_path, root, repo_id, revision, token, remote_files[setting_remote_path].expected_size)\n        setting = load_json(local_setting_path)\n\n    config = get_f5_config(manifest, setting)\n    package = VoicePackage(\n        repo_id=repo_id,\n        revision=revision,\n        root=root,\n        voice_dir=voice_dir,\n        manifest_path=local_manifest_path,\n        manifest=manifest,\n        checkpoint=checkpoint,\n        vocab_remote_path=vocab_remote_path,\n        base_vocab_remote_path=base_vocab_remote_path,\n        reference_audio_remote_path=reference_audio_remote_path,\n        reference_text_remote_path=reference_text_remote_path,\n        base_library_dir=base_library_dir,\n        setting_remote_path=setting_remote_path if setting_remote_path in remote_files else None,\n        config=config,\n        remote_files=remote_files,\n        local_setting_path=local_setting_path,\n    )\n    download_required_artifacts(package, token=token, download_checkpoint=download_checkpoint)\n    return package\n\n\ndef download_required_artifacts(package: VoicePackage, token: str | None = None, download_checkpoint: bool = True) -> VoicePackage:\n    token = token or get_hf_token()\n    package.local_vocab_path = download_remote_file(\n        package.vocab_remote_path,\n        package.root,\n        package.repo_id,\n        package.revision,\n        token,\n        package.remote_files[package.vocab_remote_path].expected_size,\n    )\n    if package.base_vocab_remote_path:\n        package.local_base_vocab_path = download_remote_file(\n            package.base_vocab_remote_path,\n            package.root,\n            package.repo_id,\n            package.revision,\n            token,\n            package.remote_files[package.base_vocab_remote_path].expected_size,\n        )\n    package.local_reference_audio_path = download_remote_file(\n        package.reference_audio_remote_path,\n        package.root,\n        package.repo_id,\n        package.revision,\n        token,\n        package.remote_files[package.reference_audio_remote_path].expected_size,\n    )\n    if package.reference_text_remote_path:\n        package.local_reference_text_path = download_remote_file(\n            package.reference_text_remote_path,\n            package.root,\n            package.repo_id,\n            package.revision,\n            token,\n            package.remote_files[package.reference_text_remote_path].expected_size,\n        )\n    else:\n        LOGGER.warning(\"Texto exato da referencia nao existe no pacote da voz; F5-TTS tentara transcrever com ASR.\")\n\n    if download_checkpoint:\n        package.local_checkpoint_path = download_remote_file(\n            package.checkpoint.remote_path,\n            package.root,\n            package.repo_id,\n            package.revision,\n            token,\n            package.checkpoint.expected_size,\n        )\n    else:\n        package.local_checkpoint_path = package.root / package.checkpoint.remote_path\n    return package\n\n\ndef read_vocab(path: Path) -> list[str]:\n    text = path.read_text(encoding=\"utf-8\")\n    return [line.rstrip(\"\\n\\r\") for line in text.splitlines()]\n\n\ndef validate_vocab(vocab_path: Path, base_vocab_path: Path | None = None) -> dict[str, Any]:\n    tokens = read_vocab(vocab_path)\n    non_empty = [token for token in tokens if token != \"\"]\n    duplicates = sorted({token for token in non_empty if non_empty.count(token) > 1})\n    chars = set(\"\".join(non_empty))\n    missing_pt = sorted(char for char in PORTUGUESE_REQUIRED_CHARS if char not in chars)\n    report: dict[str, Any] = {\n        \"path\": str(vocab_path),\n        \"tokens\": len(tokens),\n        \"empty_lines\": len(tokens) - len(non_empty),\n        \"duplicates\": duplicates,\n        \"missing_portuguese_chars\": missing_pt,\n        \"utf8\": True,\n    }\n    if base_vocab_path and base_vocab_path.exists():\n        base_tokens = read_vocab(base_vocab_path)\n        report[\"base_path\"] = str(base_vocab_path)\n        report[\"base_tokens\"] = len(base_tokens)\n        report[\"same_as_base\"] = tokens == base_tokens\n        report[\"only_voice\"] = sorted(set(tokens) - set(base_tokens))[:50]\n        report[\"only_base\"] = sorted(set(base_tokens) - set(tokens))[:50]\n    LOGGER.info(\"Relatorio vocabulario: %s\", json.dumps(report, ensure_ascii=False, indent=2))\n    if not non_empty:\n        raise RuntimeError(f\"Vocabulario vazio: {vocab_path}\")\n    if duplicates:\n        raise RuntimeError(f\"Vocabulario contem tokens duplicados: {duplicates[:10]}\")\n    return report\n\n\ndef safe_torch_load(path: Path, map_location: str = \"cpu\") -> Any:\n    import torch\n\n    try:\n        return torch.load(path, map_location=map_location, weights_only=True)\n    except TypeError:\n        return torch.load(path, map_location=map_location)\n\n\ndef tensor_shape(value: Any) -> tuple[int, ...] | None:\n    shape = getattr(value, \"shape\", None)\n    if shape is None:\n        return None\n    return tuple(int(part) for part in shape)\n\n\ndef inspect_checkpoint(checkpoint_path: Path, vocab_path: Path | None = None) -> dict[str, Any]:\n    suffix = checkpoint_path.suffix.lower()\n    if suffix == \".safetensors\":\n        from safetensors.torch import load_file\n\n        checkpoint = load_file(str(checkpoint_path), device=\"cpu\")\n    else:\n        checkpoint = safe_torch_load(checkpoint_path, map_location=\"cpu\")\n\n    info: dict[str, Any] = {\n        \"path\": str(checkpoint_path),\n        \"object_type\": type(checkpoint).__name__,\n        \"top_level_keys\": [],\n        \"has_ema\": False,\n        \"has_model_state\": False,\n        \"has_optimizer\": False,\n        \"has_scheduler\": False,\n        \"has_scaler\": False,\n        \"step\": None,\n        \"epoch\": None,\n        \"selected_weight_key\": None,\n        \"text_embedding_shape\": None,\n        \"vocab_tokens\": None,\n        \"vocab_compatible\": None,\n    }\n    if isinstance(checkpoint, dict):\n        info[\"top_level_keys\"] = sorted(str(key) for key in checkpoint.keys())[:100]\n        info[\"has_ema\"] = any(key in checkpoint for key in (\"ema_model_state_dict\", \"ema_model\"))\n        info[\"has_model_state\"] = any(key in checkpoint for key in (\"model_state_dict\", \"state_dict\", \"model\"))\n        info[\"has_optimizer\"] = any(\"optim\" in str(key).lower() for key in checkpoint)\n        info[\"has_scheduler\"] = any(\"sched\" in str(key).lower() for key in checkpoint)\n        info[\"has_scaler\"] = any(\"scaler\" in str(key).lower() for key in checkpoint)\n        info[\"step\"] = checkpoint.get(\"step\") or checkpoint.get(\"global_step\") or checkpoint.get(\"update\")\n        info[\"epoch\"] = checkpoint.get(\"epoch\")\n        state = select_inference_state_dict(checkpoint)\n        info[\"selected_weight_key\"] = state[0]\n        for key, value in state[1].items():\n            if key.endswith(\"text_embed.weight\") or \"text_embed\" in key and key.endswith(\".weight\"):\n                info[\"text_embedding_shape\"] = tensor_shape(value)\n                break\n    else:\n        raise RuntimeError(f\"Checkpoint possui tipo inesperado para inferencia: {type(checkpoint).__name__}\")\n\n    if vocab_path:\n        tokens = read_vocab(vocab_path)\n        info[\"vocab_tokens\"] = len(tokens)\n        shape = info[\"text_embedding_shape\"]\n        if shape:\n            info[\"vocab_compatible\"] = shape[0] in (len(tokens), len(tokens) + 1, len(tokens) + 2)\n\n    LOGGER.info(\"Inspecao do checkpoint: %s\", json.dumps(info, ensure_ascii=False, default=str, indent=2))\n    return info\n\n\ndef select_inference_state_dict(checkpoint: dict[str, Any]) -> tuple[str, dict[str, Any]]:\n    candidates = (\n        \"ema_model_state_dict\",\n        \"ema_model\",\n        \"model_state_dict\",\n        \"state_dict\",\n        \"model\",\n    )\n    for key in candidates:\n        value = checkpoint.get(key)\n        if isinstance(value, dict) and value:\n            if key.startswith(\"ema\"):\n                cleaned = {str(k).replace(\"ema_model.\", \"\"): v for k, v in value.items() if k not in (\"initted\", \"step\")}\n                return key, cleaned\n            return key, value\n    if checkpoint and all(hasattr(value, \"shape\") for value in checkpoint.values()):\n        return \"root_state_dict\", checkpoint\n    raise RuntimeError(\"Nao encontrei pesos de inferencia em state_dict/model_state_dict/ema_model_state_dict.\")\n\n\ndef validate_reference_audio(audio_path: Path, expected_sample_rate: int = 24000) -> dict[str, Any]:\n    import soundfile as sf\n\n    info = sf.info(str(audio_path))\n    data, sr = sf.read(str(audio_path), always_2d=True)\n    duration = len(data) / sr if sr else 0.0\n    mono = data.mean(axis=1)\n    peak = float(abs(mono).max()) if len(mono) else 0.0\n    rms = float(math.sqrt(float((mono * mono).mean()))) if len(mono) else 0.0\n    report = {\n        \"path\": str(audio_path),\n        \"format\": info.format,\n        \"sample_rate\": sr,\n        \"channels\": data.shape[1],\n        \"duration\": duration,\n        \"peak\": peak,\n        \"rms\": rms,\n        \"converted_path\": None,\n    }\n    if duration <= 0 or peak <= 0 or rms <= 1e-5:\n        raise RuntimeError(f\"Audio de referencia invalido ou silencioso: {report}\")\n    if sr != expected_sample_rate or data.shape[1] != 1:\n        converted = audio_path.with_name(f\"{audio_path.stem}_{expected_sample_rate}hz_mono.wav\")\n        try:\n            import librosa\n\n            mono_24k, _ = librosa.load(str(audio_path), sr=expected_sample_rate, mono=True)\n        except Exception:\n            mono_24k = mono\n        sf.write(str(converted), mono_24k, expected_sample_rate)\n        report[\"converted_path\"] = str(converted)\n    LOGGER.info(\"Referencia de audio: %s\", json.dumps(report, ensure_ascii=False, indent=2))\n    return report\n\n\ndef read_reference_text(path: Path | None) -> str:\n    if not path or not path.exists():\n        return \"\"\n    return path.read_text(encoding=\"utf-8\", errors=\"ignore\").strip()\n\n\ndef detect_device(device: str = \"auto\") -> str:\n    if device != \"auto\":\n        return device\n    import torch\n\n    if torch.cuda.is_available():\n        return \"cuda\"\n    if hasattr(torch, \"xpu\") and torch.xpu.is_available():\n        return \"xpu\"\n    if getattr(torch.backends, \"mps\", None) and torch.backends.mps.is_available():\n        return \"mps\"\n    return \"cpu\"\n\n\ndef generate_speech(\n    text: str,\n    output_path: str,\n    checkpoint_path: str,\n    vocab_path: str,\n    ref_audio_path: str,\n    ref_text: str,\n    device: str = \"auto\",\n    model_name: str = \"F5TTS_v1_Base\",\n    model_config: dict[str, Any] | None = None,\n) -> str:\n    text = (text or \"\").strip()\n    if not text:\n        raise ValueError(\"Texto vazio para sintese.\")\n    output = Path(output_path)\n    output.parent.mkdir(parents=True, exist_ok=True)\n\n    import numpy as np\n    import soundfile as sf\n    import torch\n    from f5_tts.infer.utils_infer import infer_process, load_model, load_vocoder, preprocess_ref_audio_text\n    from hydra.utils import get_class\n\n    selected_device = detect_device(device)\n    config = model_config or DEFAULT_F5TTS_V1_BASE_CONFIG\n    if model_name != \"F5TTS_v1_Base\":\n        raise RuntimeError(f\"Modelo F5-TTS nao suportado por este pacote: {model_name}\")\n\n    model_cls = get_class(f\"f5_tts.model.{config['backbone']}\")\n    mel_spec = config[\"mel_spec\"]\n    LOGGER.info(\"Dispositivo: %s\", selected_device)\n    LOGGER.info(\"Arquitetura: %s %s\", model_name, json.dumps(config[\"arch\"], ensure_ascii=False))\n    LOGGER.info(\"Tokenizer: %s\", config[\"tokenizer\"])\n    LOGGER.info(\"Vocoder: %s\", mel_spec[\"mel_spec_type\"])\n\n    try:\n        vocoder = load_vocoder(vocoder_name=mel_spec[\"mel_spec_type\"], device=selected_device)\n    except Exception as exc:\n        LOGGER.error(\"Falha ao carregar vocoder. Verifique internet/cache do Kaggle.\")\n        LOGGER.error(\"Traceback completo:\\n%s\", \"\".join(traceback.format_exception(type(exc), exc, exc.__traceback__)))\n        raise\n\n    use_ema = True\n    ckpt_info = inspect_checkpoint(Path(checkpoint_path), Path(vocab_path))\n    if not ckpt_info.get(\"has_ema\"):\n        use_ema = False\n    LOGGER.info(\"Uso de EMA: %s\", use_ema)\n\n    model = load_model(\n        model_cls,\n        config[\"arch\"],\n        checkpoint_path,\n        mel_spec_type=mel_spec[\"mel_spec_type\"],\n        vocab_file=vocab_path,\n        use_ema=use_ema,\n        device=selected_device,\n    )\n\n    processed_ref_audio, processed_ref_text = preprocess_ref_audio_text(ref_audio_path, ref_text, show_info=LOGGER.info)\n    final_wave, final_sample_rate, _ = infer_process(\n        processed_ref_audio,\n        processed_ref_text,\n        text,\n        model,\n        vocoder,\n        mel_spec_type=mel_spec[\"mel_spec_type\"],\n        cross_fade_duration=0.15,\n        nfe_step=32,\n        speed=1.0,\n        show_info=LOGGER.info,\n        device=selected_device,\n    )\n    if final_wave is None or len(final_wave) == 0:\n        raise RuntimeError(\"Inferencia retornou audio vazio.\")\n\n    final_wave = np.asarray(final_wave, dtype=np.float32)\n    peak = float(np.max(np.abs(final_wave)))\n    rms = float(np.sqrt(np.mean(final_wave * final_wave)))\n    duration = len(final_wave) / float(final_sample_rate)\n    if peak <= 0 or rms <= 1e-5:\n        raise RuntimeError(f\"Audio gerado parece silencioso: peak={peak}, rms={rms}\")\n\n    sf.write(str(output), final_wave, final_sample_rate)\n    validate_downloaded_file(output)\n    LOGGER.info(\"Audio gerado: %s\", output)\n    LOGGER.info(\"Duracao gerada: %.3fs | peak=%.6f | rms=%.6f\", duration, peak, rms)\n\n    del model\n    del vocoder\n    if selected_device == \"cuda\":\n        torch.cuda.empty_cache()\n    return str(output)\n\n\ndef audit_voice_package(download_checkpoint: bool = False) -> tuple[VoicePackage, DiagnosticReport]:\n    package = build_voice_package(download_checkpoint=download_checkpoint)\n    report = DiagnosticReport()\n\n    report.add(\"Manifesto\", True, str(package.manifest_path))\n    report.add(\"Checkpoint principal\", package.checkpoint.remote_path in package.remote_files, package.checkpoint.remote_path)\n    report.add(\"Arquitetura identificada\", package.manifest.get(\"architecture\") == \"F5-TTS\", package.manifest.get(\"exp_name\", \"\"))\n    report.add(\"Vocab encontrado\", bool(package.local_vocab_path and package.local_vocab_path.exists()), str(package.local_vocab_path))\n\n    try:\n        vocab_report = validate_vocab(package.local_vocab_path, package.local_base_vocab_path)\n        report.add(\"Vocab compativel\", not vocab_report.get(\"duplicates\"), f\"{vocab_report['tokens']} tokens\")\n    except Exception as exc:\n        report.add(\"Vocab compativel\", False, str(exc))\n\n    report.add(\"Referencia de audio\", bool(package.local_reference_audio_path and package.local_reference_audio_path.exists()), str(package.local_reference_audio_path))\n    ref_text = read_reference_text(package.local_reference_text_path)\n    report.add(\"Texto da referencia\", bool(ref_text), \"arquivo encontrado\" if ref_text else \"ausente; ASR sera usado\")\n    report.add(\"Vocoder\", package.config[\"mel_spec\"][\"mel_spec_type\"] == \"vocos\", package.config[\"mel_spec\"][\"mel_spec_type\"])\n\n    try:\n        import torch\n\n        report.add(\"CUDA\", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else \"CPU disponivel\")\n    except Exception as exc:\n        report.add(\"CUDA\", False, str(exc))\n\n    if download_checkpoint and package.local_checkpoint_path and package.local_checkpoint_path.exists():\n        try:\n            inspect_checkpoint(package.local_checkpoint_path, package.local_vocab_path)\n            report.add(\"Checkpoint legivel\", True, str(package.local_checkpoint_path))\n        except Exception as exc:\n            report.add(\"Checkpoint legivel\", False, str(exc))\n    else:\n        report.add(\"Checkpoint legivel\", False, \"nao baixado nesta auditoria leve\")\n\n    LOGGER.info(\"\\n%s\", report.render())\n    print(report.render())\n    return package, report\n\n\ndef report_training_artifacts(package: VoicePackage) -> dict[str, Any]:\n    files = package.remote_files\n    inference_required = {\n        \"checkpoint de inferencia\": package.checkpoint.remote_path,\n        \"vocabulario correspondente\": package.vocab_remote_path,\n        \"configuracao da arquitetura\": \"recuperada de F5TTS_v1_Base no runtime f5-tts\",\n        \"tokenizer\": package.manifest.get(\"tokenizer\"),\n        \"audio de referencia\": package.reference_audio_remote_path,\n        \"transcricao da referencia\": package.reference_text_remote_path,\n        \"vocoder e parametros de audio\": \"F5TTS_v1_Base/vocos/24000Hz/100mel\",\n    }\n    training_required = {\n        \"checkpoint completo\": package.checkpoint.remote_path,\n        \"estado do otimizador\": None,\n        \"scheduler\": None,\n        \"scaler AMP\": None,\n        \"pesos EMA\": \"a confirmar pela inspecao do checkpoint\",\n        \"contador epoch/update\": \"a confirmar pela inspecao do checkpoint\",\n        \"configuracao completa do treinamento\": package.setting_remote_path,\n        \"seed\": None,\n        \"commit F5-TTS\": None,\n        \"versoes das dependencias\": None,\n        \"dataset\": package.manifest.get(\"base_library\", {}).get(\"repo_id\"),\n        \"metadata/raw.arrow\": None,\n        \"duration.json\": f\"{package.voice_dir}/docs/duration.json\" if f\"{package.voice_dir}/docs/duration.json\" in files else None,\n        \"audios de treino\": None,\n        \"train/validation split\": None,\n        \"vocabulario\": package.vocab_remote_path,\n        \"logs e metricas\": f\"{package.voice_dir}/docs/duration.json\" if f\"{package.voice_dir}/docs/duration.json\" in files else None,\n        \"checkpoint-base exato\": f\"{package.base_library_dir}/{package.manifest.get('base_library', {}).get('checkpoint')}\" if package.base_library_dir else None,\n    }\n    report = {\n        \"obrigatorios_inferencia\": inference_required,\n        \"treinamento_reproducao\": training_required,\n        \"presentes\": sorted(path for path in files),\n        \"ausentes_inferencia\": [key for key, value in inference_required.items() if not value],\n        \"ausentes_treinamento\": [key for key, value in training_required.items() if not value],\n    }\n    LOGGER.info(\"Relatorio de artefatos: %s\", json.dumps(report, ensure_ascii=False, indent=2))\n    return report\n\n\ndef prepare_model_files(download: bool = True) -> VoicePackage:\n    return build_voice_package(download_checkpoint=download)\n\n\ndef synthesize_for_notebook(text: str = TEST_TEXT, output_path: str = \"/kaggle/working/resultado_voz.wav\") -> str:\n    package = build_voice_package(download_checkpoint=True)\n    validate_vocab(package.local_vocab_path, package.local_base_vocab_path)\n    validate_reference_audio(package.local_reference_audio_path, package.config[\"mel_spec\"][\"target_sample_rate\"])\n    ref_text = read_reference_text(package.local_reference_text_path)\n    return generate_speech(\n        text=text,\n        output_path=output_path,\n        checkpoint_path=str(package.local_checkpoint_path),\n        vocab_path=str(package.local_vocab_path),\n        ref_audio_path=str(package.local_reference_audio_path),\n        ref_text=ref_text,\n        model_name=package.manifest.get(\"exp_name\", \"F5TTS_v1_Base\"),\n        model_config=package.config,\n    )\n\n\ndef create_gradio_app():\n    import gradio as gr\n\n    package_holder: dict[str, VoicePackage] = {}\n\n    def generate(text: str):\n        try:\n            if \"package\" not in package_holder:\n                package_holder[\"package\"] = build_voice_package(download_checkpoint=True)\n            package = package_holder[\"package\"]\n            output_path = OUTPUT_DIR / \"resultado_voz_gradio.wav\"\n            audio_path = generate_speech(\n                text=text,\n                output_path=str(output_path),\n                checkpoint_path=str(package.local_checkpoint_path),\n                vocab_path=str(package.local_vocab_path),\n                ref_audio_path=str(package.local_reference_audio_path),\n                ref_text=read_reference_text(package.local_reference_text_path),\n                model_name=package.manifest.get(\"exp_name\", \"F5TTS_v1_Base\"),\n                model_config=package.config,\n            )\n            return audio_path, audio_path, f\"Audio gerado: {audio_path}\"\n        except Exception as exc:\n            LOGGER.exception(\"Erro no Gradio\")\n            return None, None, f\"Erro: {exc}\"\n\n    with gr.Blocks(title=\"Super Voz F5-TTS\") as demo:\n        gr.Markdown(\"# Super Voz F5-TTS\")\n        text_box = gr.Textbox(label=\"Texto\", value=TEST_TEXT, lines=3)\n        button = gr.Button(\"Gerar WAV\")\n        audio = gr.Audio(label=\"Audio gerado\", type=\"filepath\")\n        download = gr.File(label=\"Download\")\n        status = gr.Textbox(label=\"Status\", interactive=False)\n        button.click(generate, inputs=text_box, outputs=[audio, download, status])\n    return demo\n\n\ndef main() -> None:\n    package, report = audit_voice_package(download_checkpoint=True)\n    report_training_artifacts(package)\n    if not report.ready:\n        raise RuntimeError(\"Auditoria falhou; inferencia nao iniciada.\")\n    audio_path = synthesize_for_notebook(TEST_TEXT, \"/kaggle/working/resultado_voz.wav\")\n    print(f\"Audio gerado: {audio_path}\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
    module_path = Path('/kaggle/working/conversor_voz_kaggle.py')
    module_path.write_text(MODULE_CODE, encoding='utf-8')
    if '/kaggle/working' not in sys.path:
        sys.path.insert(0, '/kaggle/working')
    print('Modulo criado em', module_path)
except Exception as exc:
    log_error('celula 2 - criar modulo', exc)
    raise


In [ ]:
# 3) Auditoria leve: nao baixa checkpoint gigante
from pathlib import Path
import sys
import traceback

LOG_PATH = Path('/kaggle/working/super_voz_kaggle.log')

def log_error(stage, exc):
    with LOG_PATH.open('a', encoding='utf-8') as log_file:
        log_file.write(f'\n===== {stage} =====\n')
        log_file.write(''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)))
    print(f'Erro em {stage}. Log salvo em: {LOG_PATH}')

try:
    if '/kaggle/working' not in sys.path:
        sys.path.insert(0, '/kaggle/working')
    from conversor_voz_kaggle import audit_voice_package, report_training_artifacts

    package, report = audit_voice_package(download_checkpoint=False)
    artifacts_report = report_training_artifacts(package)
except Exception as exc:
    log_error('celula 3 - auditoria leve', exc)
    raise


In [ ]:
# 4) Baixar checkpoint, carregar F5-TTS e gerar WAV
from pathlib import Path
import sys
import traceback

LOG_PATH = Path('/kaggle/working/super_voz_kaggle.log')

def log_error(stage, exc):
    with LOG_PATH.open('a', encoding='utf-8') as log_file:
        log_file.write(f'\n===== {stage} =====\n')
        log_file.write(''.join(traceback.format_exception(type(exc), exc, exc.__traceback__)))
    print(f'Erro em {stage}. Log salvo em: {LOG_PATH}')

try:
    if '/kaggle/working' not in sys.path:
        sys.path.insert(0, '/kaggle/working')
    from IPython.display import Audio, FileLink, display
    from conversor_voz_kaggle import synthesize_for_notebook

    texto = 'Boa noite Warllem, sua voz esta pronta.'
    audio_path = synthesize_for_notebook(texto, '/kaggle/working/resultado_voz.wav')
    display(Audio(audio_path))
    display(FileLink(audio_path, result_html_prefix='Download do WAV: '))
    audio_path
except Exception as exc:
    log_error('celula 4 - baixar/carregar/gerar', exc)
    raise


In [ ]:
# 5) Gerar outro audio sem reexecutar o notebook inteiro
from IPython.display import Audio, FileLink, display
from conversor_voz_kaggle import synthesize_for_notebook

texto = 'Digite aqui outro texto para gerar com a voz treinada.'
audio_path = synthesize_for_notebook(texto, '/kaggle/working/resultado_voz_2.wav')
display(Audio(audio_path))
display(FileLink(audio_path, result_html_prefix='Download do WAV: '))
audio_path


## Interface opcional

A celula abaixo abre o Gradio e fica rodando enquanto a interface estiver ativa.


In [ ]:
# 6) Opcional: abrir interface Gradio
from conversor_voz_kaggle import create_gradio_app

demo = create_gradio_app()
demo.launch(share=True, debug=True)
